# Notebook 16 - Uji Coba LSTM dengan Label Triple-Barrier (Beberapa Nilai k)

**Notebook 13 TIDAK diubah** (tetap jadi rujukan resmi). Notebook ini
sebuah uji coba terisolasi: melatih `FlexibleLSTMClassifier` **sesungguhnya**
memakai skema label **triple-barrier decisive-only**, dicoba untuk beberapa
nilai `k` (`k=1.0` dan `k=0.5`), masing-masing dibandingkan langsung
terhadap hasil resmi Notebook 13 (label naive, `data/artifacts/lstm_weights/
notebook13_fixed_metadata.json`) DAN didiagnosis akar penyebabnya (audit
data + uji kapasitas + uji permutasi) - metodologi yang sama dipakai utk
tiap k, lewat fungsi yang di-reuse (bukan copy-paste per k).

**Yang di-reuse dari Notebook 13 (supaya perbandingan adil):** `best_config`
(TIDAK di-random-search ulang), resep training yang sudah diperbaiki (early
stopping monitor `val_f1_macro`, `min_epochs=20`, `patience=15`), 6 seed
ensemble yang SAMA `[2, 16, 25, 27, 32, 64]`, split kronologis 70/15/15,
fitur (`RETURN_FEATURE_COLS_B`), window=20, horizon=5.

**Ringkasan hasil k=1.0 (sudah diuji sebelumnya)**: AUC test 0.556 (kalah
dari naive 0.645), TIDAK collapse, tapi `train_loss` mentok di 0.61.
Diagnosis: BUKAN bug preprocessing/kapasitas (uji kapasitas -> train_loss
turun ke 0.0001 kalau dibiarkan lama), tapi uji permutasi menunjukkan pada
443 sampel training yang tersisa, sinyal fitur TIDAK signifikan secara
statistik (p=0.46) - k=1.0 kemungkinan terlalu agresif membuang data.

## Setup

In [10]:
from __future__ import annotations

import json
import logging
import sys
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("E:/forecasting_index_snp500/stock_prediction")
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("notebook16")

ARTIFACT_DIR = ROOT / "data/artifacts/lstm_weights"
NAIVE_METADATA_PATH = ARTIFACT_DIR / "notebook13_fixed_metadata.json"  # HANYA DIBACA

OUT_DIR = ROOT / "notebooks" / "outputs" / "notebook16"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 20
HORIZON = 5
SEED = 42
ENSEMBLE_SEEDS = [2, 16, 25, 27, 32, 64]  # SAMA PERSIS dgn Notebook 13

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", device)

from pipeline.feature_engineering.dataset_builder import load_return_features, RETURN_FEATURE_COLS_B
from models.lstm.notebook07_common import compute_iqr_bounds, apply_capping, _make_windows
from models.lstm.notebook07_common import compute_class_weights, _train_one_epoch, _evaluate_epoch, sweep_best_threshold
from models.lstm.lstm_model import FlexibleLSTMClassifier

FEATURE_COLS = RETURN_FEATURE_COLS_B
CAPPING_COLS = RETURN_FEATURE_COLS_B

with open(NAIVE_METADATA_PATH, encoding="utf-8") as f:
    naive_metadata = json.load(f)
best_config = naive_metadata["config"]
MIN_EPOCHS = naive_metadata["training_recipe_fix"]["min_epochs"]
PATIENCE = naive_metadata["training_recipe_fix"]["early_stopping_patience"]
assert naive_metadata["feature_cols"] == FEATURE_COLS
assert naive_metadata["window"] == WINDOW and naive_metadata["horizon"] == HORIZON

print("best_config (REUSE dari Notebook 13):", best_config)
print(f"min_epochs={MIN_EPOCHS}  patience={PATIENCE}")

13:42:42  INFO      Device: cuda


best_config (REUSE dari Notebook 13): {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.25757937597341474, 'optimizer': 'adam', 'activation': 'tanh', 'lr': 0.002971160090697491, 'epochs': 150, 'batch_size': 32, 'weight_decay': 0.0}
min_epochs=20  patience=15


## Bagian 1 - Load Data (Sama dengan Notebook 13/15)

In [11]:
df_raw = load_return_features().sort_values("Date").reset_index(drop=True)
print(f"Total baris : {len(df_raw):,} ({df_raw['Date'].min().date()} s/d {df_raw['Date'].max().date()})")
df_raw.head()

13:42:42  INFO      Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
13:42:42  INFO      Sentiment loaded: 1528 rows (2018-01-02 – 2024-03-04)
13:42:42  INFO      Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)
13:42:42  INFO      Return-features dataset: 1526 rows (2018-01-03 – 2024-03-04)


Total baris : 1,526 (2018-01-03 s/d 2024-03-04)


,Date,ret_open,ret_high,ret_low,ret_close,ret_volume,sentiment_mean,Close
0,2018-01-03,0.005261,0.006855,0.005745,0.006399,0.043150,-0.166667,2713.060059
1,2018-01-04,0.007954,0.005497,0.007895,0.004029,0.043259,0.000000,2723.989990
2,2018-01-05,0.004420,0.005188,0.003255,0.007034,-0.123889,0.000000,2743.149902
3,2018-01-08,0.004152,0.001844,0.003549,0.001662,0.002124,0.166667,2747.709961
4,2018-01-09,0.003092,0.003868,0.003748,0.001303,0.068173,0.666667,2751.290039


## Bagian 2 - Fungsi Reusable: Label Triple-Barrier, Dataset, Training, Diagnosis

Semua fungsi di bawah dipakai UTUH untuk tiap nilai `k` (dipanggil ulang,
bukan copy-paste) - supaya perbandingan antar-k murni karena `k`, bukan
karena perbedaan kode.

In [12]:
def triple_barrier_labels(close: np.ndarray, vol: np.ndarray, horizon: int, k: float):
    n = len(close)
    labels = np.full(n, np.nan)
    touch_type = np.array([None] * n, dtype=object)
    for t in range(n - horizon):
        v = vol[t]
        if np.isnan(v) or v == 0:
            continue
        barrier = k * v * np.sqrt(horizon)
        path = close[t + 1: t + horizon + 1] / close[t] - 1.0
        upper_hits = np.flatnonzero(path >= barrier)
        lower_hits = np.flatnonzero(path <= -barrier)
        first_upper = upper_hits[0] if len(upper_hits) else np.inf
        first_lower = lower_hits[0] if len(lower_hits) else np.inf
        if first_upper == np.inf and first_lower == np.inf:
            labels[t] = 1.0 if path[-1] > 0 else 0.0
            touch_type[t] = "vertical"
        elif first_upper < first_lower:
            labels[t] = 1.0
            touch_type[t] = "upper"
        else:
            labels[t] = 0.0
            touch_type[t] = "lower"
    return labels, touch_type


def build_dataset(df_raw: pd.DataFrame, k: float) -> dict:
    """Label triple-barrier(k) -> split 70/15/15 -> capping -> scaling ->
    windowing (semua window dulu) -> filter decisive-only (setelah windowing,
    supaya kontinuitas window tetap utuh)."""
    df = df_raw.copy()
    df["vol_20"] = df["ret_close"].rolling(20).std()
    labels_tb, touch_tb = triple_barrier_labels(df["Close"].values, df["vol_20"].values, HORIZON, k)
    df["label"] = labels_tb
    df["touch_type"] = touch_tb
    df["is_decisive"] = df["touch_type"].isin(["upper", "lower"]).astype("int64")
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    df["label"] = df["label"].astype("int64")

    n = len(df)
    n_train, n_val = int(n * 0.70), int(n * 0.15)
    df_train = df.iloc[:n_train].reset_index(drop=True)
    df_val = df.iloc[n_train: n_train + n_val].reset_index(drop=True)
    df_test = df.iloc[n_train + n_val:].reset_index(drop=True)

    bounds = compute_iqr_bounds(df_train, CAPPING_COLS)
    df_train, df_val, df_test = apply_capping(df_train, bounds), apply_capping(df_val, bounds), apply_capping(df_test, bounds)

    scaler = MinMaxScaler()
    train_feat = scaler.fit_transform(df_train[FEATURE_COLS].values.astype(np.float64))
    val_feat = scaler.transform(df_val[FEATURE_COLS].values.astype(np.float64))
    test_feat = scaler.transform(df_test[FEATURE_COLS].values.astype(np.float64))

    X_train_all, y_train_all = _make_windows(train_feat, df_train["label"].values, WINDOW)
    X_val_all, y_val_all = _make_windows(val_feat, df_val["label"].values, WINDOW)
    X_test_all, y_test_all = _make_windows(test_feat, df_test["label"].values, WINDOW)
    _, dec_train = _make_windows(train_feat, df_train["is_decisive"].values, WINDOW)
    _, dec_val = _make_windows(val_feat, df_val["is_decisive"].values, WINDOW)
    _, dec_test = _make_windows(test_feat, df_test["is_decisive"].values, WINDOW)

    m_tr, m_va, m_te = dec_train.astype(bool), dec_val.astype(bool), dec_test.astype(bool)
    return {
        "k": k,
        "touch_counts": pd.Series(touch_tb).value_counts(dropna=False).to_dict(),
        "X_train": X_train_all[m_tr], "y_train": y_train_all[m_tr],
        "X_val": X_val_all[m_va], "y_val": y_val_all[m_va],
        "X_test": X_test_all[m_te], "y_test": y_test_all[m_te],
        "n_before": {"train": len(X_train_all), "val": len(X_val_all), "test": len(X_test_all)},
    }


def build_optimizer_wd(name: str, params, lr: float, weight_decay: float = 0.0):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Optimizer tidak dikenal: {name}")


def val_f1_macro_quick(model, X_val_t, y_val, n_steps: int = 39) -> float:
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(X_val_t), dim=1)[:, 1].cpu().numpy()
    best = -1.0
    for t in np.linspace(0.05, 0.95, n_steps):
        best = max(best, f1_score(y_val, (p >= t).astype(int), average="macro", zero_division=0))
    return float(best)


def train_with_early_stopping(model, train_loader, val_loader, X_val_t, y_val,
                              optimizer, criterion, device, max_epochs: int,
                              min_epochs: int = MIN_EPOCHS, patience: int = PATIENCE,
                              log_prefix: str = "", log_every: int = 50):
    best_score, best_state, epochs_no_improve, stopped_epoch = -1.0, None, 0, max_epochs
    history = {"train_loss": [], "val_f1_macro": []}
    for epoch in range(1, max_epochs + 1):
        train_loss, _ = _train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_f1m = val_f1_macro_quick(model, X_val_t, y_val)
        history["train_loss"].append(train_loss)
        history["val_f1_macro"].append(val_f1m)
        if epoch == 1 or epoch % log_every == 0 or epoch == max_epochs:
            logger.info("%sepoch %3d/%d - train_loss=%.4f val_f1_macro=%.4f", log_prefix, epoch, max_epochs, train_loss, val_f1m)
        if val_f1m > best_score + 1e-5:
            best_score = val_f1m
            best_state = deepcopy({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            epochs_no_improve = 0
        else:
            if epoch >= min_epochs:
                epochs_no_improve += 1
            if epoch >= min_epochs and epochs_no_improve >= patience:
                stopped_epoch = epoch
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, stopped_epoch


def run_ensemble(ds: dict, seeds: list[int]) -> dict:
    X_train, y_train = ds["X_train"], ds["y_train"]
    X_val, y_val = ds["X_val"], ds["y_val"]
    X_test, y_test = ds["X_test"], ds["y_test"]
    y_test_arr = y_test.astype(int)

    train_tensor = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_tensor = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    val_loader = DataLoader(val_tensor, batch_size=best_config["batch_size"], shuffle=False)
    X_val_t = torch.from_numpy(X_val).to(device)
    X_test_t = torch.from_numpy(X_test).to(device)

    models, seed_rows = [], []
    for seed in seeds:
        torch.manual_seed(seed)
        model_s = FlexibleLSTMClassifier(
            input_size=len(FEATURE_COLS), hidden_size=best_config["hidden_size"],
            num_layers=best_config["num_layers"], dropout=best_config["dropout"],
            activation=best_config["activation"], num_classes=2,
        ).to(device)
        optimizer_s = build_optimizer_wd(best_config["optimizer"], model_s.parameters(), best_config["lr"], best_config["weight_decay"])
        class_weights_s = compute_class_weights(y_train, device)
        criterion_s = nn.CrossEntropyLoss(weight=class_weights_s)
        train_loader_s = DataLoader(train_tensor, batch_size=best_config["batch_size"], shuffle=True)

        history_s, stopped_epoch_s = train_with_early_stopping(
            model_s, train_loader_s, val_loader, X_val_t, y_val,
            optimizer_s, criterion_s, device, max_epochs=best_config["epochs"],
            log_prefix=f"  [k={ds['k']} seed={seed}] ",
        )
        models.append(model_s)
        seed_rows.append({"seed": seed, "stopped_epoch": stopped_epoch_s, "min_train_loss": float(min(history_s["train_loss"]))})
        logger.info("k=%s seed=%d done (stop epoch %d, min_train_loss=%.4f)", ds["k"], seed, stopped_epoch_s, seed_rows[-1]["min_train_loss"])

    with torch.no_grad():
        ens_val_probs = np.mean([torch.softmax(m(X_val_t), dim=1)[:, 1].cpu().numpy() for m in models], axis=0)
        ens_test_probs = np.mean([torch.softmax(m(X_test_t), dim=1)[:, 1].cpu().numpy() for m in models], axis=0)
    threshold, val_f1_macro = sweep_best_threshold(y_val, ens_val_probs)
    y_test_pred = (ens_test_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_arr, y_test_pred, labels=[0, 1]).ravel()

    metrics = {
        "accuracy": accuracy_score(y_test_arr, y_test_pred),
        "precision": precision_score(y_test_arr, y_test_pred, zero_division=0),
        "recall": recall_score(y_test_arr, y_test_pred, zero_division=0),
        "f1": f1_score(y_test_arr, y_test_pred, zero_division=0),
        "f1_macro": f1_score(y_test_arr, y_test_pred, average="macro", zero_division=0),
    }
    diagnostics = {
        "auc_test": roc_auc_score(y_test_arr, ens_test_probs) if len(set(y_test_arr)) > 1 else float("nan"),
        "bal_acc_test": balanced_accuracy_score(y_test_arr, y_test_pred),
        "std_prob_up": float(ens_test_probs.std()),
        "min_train_loss": float(min(r["min_train_loss"] for r in seed_rows)),
    }
    collapse = bool(tn == 0 or tp == 0 or diagnostics["std_prob_up"] < 0.02)
    return {
        "models": models, "seed_rows": seed_rows, "threshold": threshold,
        "metrics": metrics, "diagnostics": diagnostics,
        "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "collapse": collapse, "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test),
    }


def run_diagnosis(ds: dict, capacity_epochs: int = 300, n_perm: int = 50) -> dict:
    X_train, y_train = ds["X_train"], ds["y_train"]
    X_test, y_test = ds["X_test"], ds["y_test"]

    no_nan_inf = bool(
        np.isnan(X_train).sum() == 0 and np.isnan(X_test).sum() == 0
        and np.isinf(X_train).sum() == 0 and np.isinf(X_test).sum() == 0
    )

    train_tensor = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    train_loader_cap = DataLoader(train_tensor, batch_size=best_config["batch_size"], shuffle=True)
    torch.manual_seed(SEED)
    model_cap = FlexibleLSTMClassifier(
        input_size=len(FEATURE_COLS), hidden_size=best_config["hidden_size"],
        num_layers=best_config["num_layers"], dropout=0.0, activation=best_config["activation"], num_classes=2,
    ).to(device)
    optimizer_cap = torch.optim.Adam(model_cap.parameters(), lr=best_config["lr"], weight_decay=0.0)
    criterion_cap = nn.CrossEntropyLoss()
    cap_losses = []
    for epoch in range(1, capacity_epochs + 1):
        tl, _ = _train_one_epoch(model_cap, train_loader_cap, optimizer_cap, criterion_cap, device)
        cap_losses.append(tl)
    min_loss_cap = min(cap_losses)

    X_train_last, X_test_last = X_train[:, -1, :], X_test[:, -1, :]
    scaler_perm = StandardScaler().fit(X_train_last)
    Xtr_s, Xte_s = scaler_perm.transform(X_train_last), scaler_perm.transform(X_test_last)
    clf_real = LogisticRegression(max_iter=1000).fit(Xtr_s, y_train)
    auc_real = roc_auc_score(y_test, clf_real.predict_proba(Xte_s)[:, 1])

    rng_perm = np.random.default_rng(SEED)
    auc_null = []
    for _ in range(n_perm):
        y_perm = rng_perm.permutation(y_train)
        clf_p = LogisticRegression(max_iter=1000).fit(Xtr_s, y_perm)
        auc_null.append(roc_auc_score(y_test, clf_p.predict_proba(Xte_s)[:, 1]))
    auc_null = np.array(auc_null)
    p_value = float((auc_null >= auc_real).mean())

    return {
        "no_nan_inf": no_nan_inf, "min_loss_capacity": min_loss_cap,
        "auc_real": float(auc_real), "auc_null_mean": float(auc_null.mean()),
        "auc_null_p95": float(np.percentile(auc_null, 95)), "p_value": p_value,
        "signal_significant": p_value < 0.05,
    }

## Bagian 3 - Jalankan untuk k=1.0

In [13]:
ds_k1 = build_dataset(df_raw, k=1.0)
print(f"k=1.0 | touch_counts={ds_k1['touch_counts']}")
print(f"n_train={len(ds_k1['X_train'])}  n_val={len(ds_k1['X_val'])}  n_test={len(ds_k1['X_test'])}")

ens_k1 = run_ensemble(ds_k1, ENSEMBLE_SEEDS)
diag_k1 = run_diagnosis(ds_k1)

print("\n=== k=1.0 - hasil ensemble ===")
print(ens_k1["metrics"])
print(ens_k1["diagnostics"])
print("collapse:", ens_k1["collapse"])
print("\n=== k=1.0 - diagnosis akar penyebab ===")
print(diag_k1)

k=1.0 | touch_counts={'vertical': 851, 'upper': 357, 'lower': 294, nan: 24}
n_train=443  n_val=84  n_test=103


13:42:42  INFO        [k=1.0 seed=2] epoch   1/150 - train_loss=0.7022 val_f1_macro=0.5088
13:42:52  INFO        [k=1.0 seed=2] epoch  50/150 - train_loss=0.6135 val_f1_macro=0.6354
13:42:52  INFO      k=1.0 seed=2 done (stop epoch 55, min_train_loss=0.5758)
13:42:52  INFO        [k=1.0 seed=16] epoch   1/150 - train_loss=0.6996 val_f1_macro=0.4122
13:42:58  INFO      k=1.0 seed=16 done (stop epoch 38, min_train_loss=0.6555)
13:42:58  INFO        [k=1.0 seed=25] epoch   1/150 - train_loss=0.6923 val_f1_macro=0.4999
13:43:05  INFO        [k=1.0 seed=25] epoch  50/150 - train_loss=0.6238 val_f1_macro=0.6876
13:43:09  INFO      k=1.0 seed=25 done (stop epoch 78, min_train_loss=0.4780)
13:43:10  INFO        [k=1.0 seed=27] epoch   1/150 - train_loss=0.6982 val_f1_macro=0.4911
13:43:17  INFO        [k=1.0 seed=27] epoch  50/150 - train_loss=0.6294 val_f1_macro=0.5952
13:43:17  INFO      k=1.0 seed=27 done (stop epoch 51, min_train_loss=0.6117)
13:43:17  INFO        [k=1.0 seed=32] epoch   1


=== k=1.0 - hasil ensemble ===
{'accuracy': 0.5922330097087378, 'precision': 0.6875, 'recall': 0.6666666666666666, 'f1': 0.676923076923077, 'f1_macro': 0.5621457489878543}
{'auc_test': 0.6036036036036035, 'bal_acc_test': 0.5630630630630631, 'std_prob_up': 0.11077021062374115, 'min_train_loss': 0.4780273518228639}
collapse: False

=== k=1.0 - diagnosis akar penyebab ===
{'no_nan_inf': True, 'min_loss_capacity': 0.00010612707418685518, 'auc_real': 0.5012285012285012, 'auc_null_mean': 0.5013431613431613, 'auc_null_p95': 0.540069615069615, 'p_value': 0.46, 'signal_significant': False}


## Bagian 4 - Jalankan untuk k=0.5

In [14]:
ds_k05 = build_dataset(df_raw, k=0.5)
print(f"k=0.5 | touch_counts={ds_k05['touch_counts']}")
print(f"n_train={len(ds_k05['X_train'])}  n_val={len(ds_k05['X_val'])}  n_test={len(ds_k05['X_test'])}")

ens_k05 = run_ensemble(ds_k05, ENSEMBLE_SEEDS)
diag_k05 = run_diagnosis(ds_k05)

print("\n=== k=0.5 - hasil ensemble ===")
print(ens_k05["metrics"])
print(ens_k05["diagnostics"])
print("collapse:", ens_k05["collapse"])
print("\n=== k=0.5 - diagnosis akar penyebab ===")
print(diag_k05)

k=0.5 | touch_counts={'upper': 723, 'lower': 518, 'vertical': 261, nan: 24}
n_train=847  n_val=166  n_test=185


13:43:54  INFO        [k=0.5 seed=2] epoch   1/150 - train_loss=0.6961 val_f1_macro=0.3591
13:44:04  INFO        [k=0.5 seed=2] epoch  50/150 - train_loss=0.6143 val_f1_macro=0.5057
13:44:04  INFO      k=0.5 seed=2 done (stop epoch 53, min_train_loss=0.5964)
13:44:05  INFO        [k=0.5 seed=16] epoch   1/150 - train_loss=0.6973 val_f1_macro=0.3591
13:44:15  INFO      k=0.5 seed=16 done (stop epoch 44, min_train_loss=0.6652)
13:44:15  INFO        [k=0.5 seed=25] epoch   1/150 - train_loss=0.6965 val_f1_macro=0.3773
13:44:27  INFO        [k=0.5 seed=25] epoch  50/150 - train_loss=0.6400 val_f1_macro=0.4875
13:44:38  INFO        [k=0.5 seed=25] epoch 100/150 - train_loss=0.1762 val_f1_macro=0.5653
13:44:38  INFO      k=0.5 seed=25 done (stop epoch 102, min_train_loss=0.1344)
13:44:38  INFO        [k=0.5 seed=27] epoch   1/150 - train_loss=0.6946 val_f1_macro=0.3591
13:44:49  INFO        [k=0.5 seed=27] epoch  50/150 - train_loss=0.5706 val_f1_macro=0.5196
13:44:50  INFO      k=0.5 seed=2


=== k=0.5 - hasil ensemble ===
{'accuracy': 0.5783783783783784, 'precision': 0.6240601503759399, 'recall': 0.7477477477477478, 'f1': 0.680327868852459, 'f1_macro': 0.53064012490242}
{'auc_test': 0.5756026296566837, 'bal_acc_test': 0.5360360360360361, 'std_prob_up': 0.0977695882320404, 'min_train_loss': 0.13438431155970132}
collapse: False

=== k=0.5 - diagnosis akar penyebab ===
{'no_nan_inf': True, 'min_loss_capacity': 8.82503717169343e-06, 'auc_real': 0.5129047966885805, 'auc_null_mean': 0.4974141709276844, 'auc_null_p95': 0.5381665449233017, 'p_value': 0.34, 'signal_significant': False}


## Bagian 5 - Perbandingan Akhir: Naive vs Triple-Barrier k=1.0 vs k=0.5

In [15]:
naive_metrics, naive_diag, naive_cm = naive_metadata["metrics"], naive_metadata["diagnostics"], naive_metadata["confusion_matrix"]
naive_collapse = bool(naive_cm["tn"] == 0 or naive_cm["tp"] == 0 or naive_diag["std_prob_up"] < 0.02)

comparison = pd.DataFrame({
    "Naive (Notebook 13)": {
        **naive_metrics, **naive_diag,
        "n_train": naive_metadata["split_sizes"]["train_samples"],
        "n_test": naive_metadata["split_sizes"]["test_samples"],
        "collapse": naive_collapse,
    },
    "Triple-barrier k=1.0": {
        **ens_k1["metrics"], **ens_k1["diagnostics"],
        "n_train": ens_k1["n_train"], "n_test": ens_k1["n_test"], "collapse": ens_k1["collapse"],
    },
    "Triple-barrier k=0.5": {
        **ens_k05["metrics"], **ens_k05["diagnostics"],
        "n_train": ens_k05["n_train"], "n_test": ens_k05["n_test"], "collapse": ens_k05["collapse"],
    },
})

diagnosis_table = pd.DataFrame({
    "Triple-barrier k=1.0": diag_k1,
    "Triple-barrier k=0.5": diag_k05,
})

comparison_path = OUT_DIR / "final_comparison_naive_vs_k1_vs_k05.csv"
diagnosis_path = OUT_DIR / "final_diagnosis_k1_vs_k05.csv"
comparison.to_csv(comparison_path)
diagnosis_table.to_csv(diagnosis_path)

print("=== Perbandingan Metrik (Test Set) ===")
print(comparison.to_string())
print(f"\nDisimpan ke: {comparison_path}")

print("\n=== Perbandingan Diagnosis Akar Penyebab ===")
print(diagnosis_table.to_string())
print(f"\nDisimpan ke: {diagnosis_path}")

=== Perbandingan Metrik (Test Set) ===
               Naive (Notebook 13) Triple-barrier k=1.0 Triple-barrier k=0.5
accuracy                  0.542857             0.592233             0.578378
precision                 0.822581               0.6875              0.62406
recall                       0.375             0.666667             0.747748
f1                        0.515152             0.676923             0.680328
f1_macro                   0.54136             0.562146              0.53064
auc_test                  0.644972             0.603604             0.575603
bal_acc_test              0.613176             0.563063             0.536036
std_prob_up               0.105191              0.11077              0.09777
min_train_loss            0.175209             0.478027             0.134384
n_train                       1045                  443                  847
n_test                         210                  103                  185
collapse                     False   

## Penutup

Perbandingan di atas murni karena skema label & nilai `k` (hyperparameter,
resep training, 6 seed identik dengan Notebook 13). Baca metrik BERDAMPINGAN
dengan `n_train`/`collapse`/`signal_significant` (dari uji permutasi) -
angka AUC/F1 tertinggi tidak berarti apa-apa kalau `signal_significant` di
dataset itu `False` (berarti hasilnya secara statistik tidak beda dari
kebetulan). Notebook 13 TIDAK diubah oleh notebook ini.